# Fake Job Posting Detector — EDA & Model Training

Text + tabular fusion model (TF-IDF + Logistic Regression).

## 1. Load DataSet

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/fake_job_postings.csv")
print(df.shape)
df.head()

(17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


## 2. Missingness Analysis

Chi-square test (Bonferroni-corrected, α = 0.005 for 10 columns) to
determine which columns' missingness is statistically associated with
fraud. Six columns pass and get an engineered `_missing` flag preserved
as a feature — see project report Section 3.2 for full results and
interpretation.

In [2]:
from scipy.stats import chi2_contingency

candidate_cols = ['salary_range', 'department', 'required_education', 'benefits',
                   'required_experience', 'function', 'industry', 'employment_type',
                   'company_profile', 'requirements']

for col in candidate_cols:
    contingency = pd.crosstab(df[col].isnull(), df['fraudulent'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"{col}: p-value = {p:.6f}")

salary_range: p-value = 0.000000
department: p-value = 0.043126
required_education: p-value = 0.000050
benefits: p-value = 0.313496
required_experience: p-value = 0.000000
function: p-value = 0.083552
industry: p-value = 0.003836
employment_type: p-value = 0.000000
company_profile: p-value = 0.000000
requirements: p-value = 0.025651


## 3. Feature Engineering

In [3]:
# Missingness flags — only for columns that passed significance testing
missing_flag_cols = ['salary_range', 'required_experience', 'employment_type','company_profile', 'required_education', 'industry']

for col in missing_flag_cols:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

# Combine text fields into one blob for TF-IDF
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in text_cols:
    df[col] = df[col].fillna('')

df['text_combined'] = (df['title'] + ' ' + df['company_profile'] + ' ' +
                        df['description'] + ' ' + df['requirements'] + ' ' +
                        df['benefits'])

# Tabular categorical encoding
cat_cols = ['employment_type', 'required_experience', 'required_education',
            'industry', 'function', 'department']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(df_encoded.shape)
print(df_encoded.filter(like='_missing').sum())

(17880, 1548)
salary_range_missing           15012
required_experience_missing     7050
employment_type_missing         3471
company_profile_missing         3308
required_education_missing      8105
industry_missing                4903
dtype: int64


## 4. Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df_encoded.columns
                 if c not in ['job_id', 'title', 'location', 'department',
                              'salary_range', 'company_profile', 'description',
                              'requirements', 'benefits', 'text_combined', 'fraudulent']]

X_tabular = df_encoded[feature_cols]
X_text = df_encoded['text_combined']
y = df_encoded['fraudulent']

X_tab_train, X_tab_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_tabular, X_text, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

fraudulent
0    0.951552
1    0.048448
Name: proportion, dtype: float64
fraudulent
0    0.951622
1    0.048378
Name: proportion, dtype: float64


## 5. TF-IDF + Fusion

TF-IDF is fit ONLY on training text (leakage prevention) — test text is
transformed, never re-fit. Tabular features are cast to float32 before
fusion since a mixed bool/int64 DataFrame upcasts to `object` dtype on
`.values`, which scipy.sparse rejects.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf = tfidf.transform(X_text_test)

X_tab_train_numeric = X_tab_train.astype(np.float32)
X_tab_test_numeric = X_tab_test.astype(np.float32)

X_train_fused = hstack([X_text_train_tfidf, csr_matrix(X_tab_train_numeric.values)])
X_test_fused = hstack([X_text_test_tfidf, csr_matrix(X_tab_test_numeric.values)])

print("Fused train shape:", X_train_fused.shape)
print("Fused test shape:", X_test_fused.shape)

Fused train shape: (14304, 6538)
Fused test shape: (3576, 6538)


## 6. Train & Evaluate

`class_weight='balanced'` to counteract the 95/5 class imbalance.
PR-AUC is the primary metric (accuracy is misleading under this
imbalance) — see project report Section 4.5 for justification.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_curve, auc, roc_auc_score

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model.fit(X_train_fused, y_train)

y_pred = model.predict(X_test_fused)
y_proba = model.predict_proba(X_test_fused)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)
print(f"PR-AUC: {pr_auc:.4f}")

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

              precision    recall  f1-score   support

   Not Fraud       0.99      0.97      0.98      3403
       Fraud       0.61      0.90      0.72       173

    accuracy                           0.97      3576
   macro avg       0.80      0.93      0.85      3576
weighted avg       0.98      0.97      0.97      3576

PR-AUC: 0.8902
ROC-AUC: 0.9869


## 7. Save Model Artifacts

All three are required for reproducible inference — see `predict.py`.

In [7]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/logistic_regression_model.pkl")
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(feature_cols, "../models/tabular_feature_columns.pkl")

print("Saved: model, vectorizer, and feature column list to ../models/")

Saved: model, vectorizer, and feature column list to ../models/
